In [10]:
import math
from scipy.stats import entropy

In [11]:
documents = {
"doc1": "nepse analysis is essential for investors",
"doc2": "machine learning helps in predicting stock prices",
"doc3": "nepse is the stock market name of nepal"
}

In [12]:
def preprocess(text):
    return text.lower().split()


vocab = set()
preprocessed_docs = {}
for name, text in documents.items():
    tokens = preprocess(text)
    preprocessed_docs[name] = tokens
    vocab.update(tokens)

vocab = sorted(list(vocab))
print("Vocabulary:", vocab)

Vocabulary: ['analysis', 'essential', 'for', 'helps', 'in', 'investors', 'is', 'learning', 'machine', 'market', 'name', 'nepal', 'nepse', 'of', 'predicting', 'prices', 'stock', 'the']


In [13]:
def compute_tf(doc_tokens, vocab):
    tf = {}
    total_terms = len(doc_tokens)
    for term in vocab:
        tf[term] = doc_tokens.count(term) / total_terms
        return tf


def compute_idf(all_docs, vocab):
    N = len(all_docs)
    idf = {}
    for term in vocab:
        df = sum(1 for doc in all_docs.values() if term in doc)
        idf[term] = math.log(N / (df + 1)) + 1 # smoothing
    return idf


def compute_tfidf(tf, idf):
    return {term: tf[term] * idf[term] for term in tf}


idf = compute_idf(preprocessed_docs, vocab)


for name, tokens in preprocessed_docs.items():
    tf = compute_tf(tokens, vocab)
    tfidf = compute_tfidf(tf, idf)
    print(f"\nDocument: {name}")
    print("TF: ", {k: round(v, 3) for k, v in tf.items()})
    print("TF-IDF: ", {k: round(v, 3) for k, v in tfidf.items()})


Document: doc1
TF:  {'analysis': 0.167}
TF-IDF:  {'analysis': 0.234}

Document: doc2
TF:  {'analysis': 0.0}
TF-IDF:  {'analysis': 0.0}

Document: doc3
TF:  {'analysis': 0.0}
TF-IDF:  {'analysis': 0.0}


In [14]:
def bow_vector(doc, vocab):
    words = doc.lower().split()
    return [words.count(term) for term in vocab]


def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = math.sqrt(sum(a * a for a in vec1))
    magnitude2 = math.sqrt(sum(b * b for b in vec2))
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)


vectors = [bow_vector(doc, vocab) for doc in documents.values()]


print("\nCosine Similarities:")
for i in range(len(vectors)):
    for j in range(i+1, len(vectors)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"Doc{i+1} vs Doc{j+1}: {round(sim, 4)}")


Cosine Similarities:
Doc1 vs Doc2: 0.0
Doc1 vs Doc3: 0.2887
Doc2 vs Doc3: 0.1336


In [15]:
def normalize(vec):
    total = sum(vec)
    if total == 0:
        return [0 for _ in vec]
    return [v/total for v in vec]


print("\nKL Divergences (lower = closer):")
Q = vectors[0] # take Doc1 as query example
Q_prob = normalize(Q)
for i, d in enumerate(vectors[1:], 2):
    D_prob = normalize(d)
    epsilon = 1e-10
    div = entropy([p+epsilon for p in Q_prob], [q+epsilon for q in D_prob])
    print(f"Doc1 vs Doc{i}: {round(div, 4)}")


KL Divergences (lower = closer):
Doc1 vs Doc2: 21.2341
Doc1 vs Doc3: 14.252
